# Lab 5: Galaxy Image Generation with Diffusion Models

In this lab, you'll build a **denoising diffusion probabilistic model (DDPM)** to generate galaxy images from scratch. Instead of reconstructing galaxies through a bottleneck (Lab 4), you'll learn to iteratively **denoise** them — starting from pure noise and refining step by step.

**Your tasks** (in `diffusion.py`):
1. Implement the **noise schedule** — the sequence of noise levels $\beta_t$
2. Implement the **forward process** — the closed-form noising $q(z_t \mid x)$
3. Build a **noise predictor network** `EpsNet` — a time-conditioned CNN that predicts $\epsilon$
4. Write the **training loss** — the denoising objective

**Given** (do not modify): `sinusoidal_embedding`, `ddpm_sample_step`, `train_step`, `save_model` / `load_model`

**Grading (4 points):**
- 1 pt: Noise schedule and forward process are correct
- 1 pt: Loss decreases during training
- 1 pt: Saved model generates valid samples
- 1 pt: Generated galaxies have radial structure (bright center, dark edges)

In [ ]:
# Uncomment and run this cell if you are on Google Colab
# !pip install jax jaxlib flax optax

In [ ]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
import optax

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12

from diffusion import IMG_SHAPE

---
## Part A: The Data

Same galaxy images from Labs 3 and 4. In Lab 3 you classified them; in Lab 4 you reconstructed them with a VAE. Now you'll learn to **generate** them from pure noise using diffusion.

In [ ]:
data = np.load("galaxy_data.npz")

X_train = data["X_train"].astype(np.float32) / 255.0
X_val = data["X_val"].astype(np.float32) / 255.0
X_test = data["X_test"].astype(np.float32) / 255.0

print(f"Train: {X_train.shape}  (values in [{X_train.min():.1f}, {X_train.max():.1f}])")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap="gray")
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Sample Galaxy Images (32x32 grayscale)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part B: Noise Schedule (TODO #1)

Diffusion models add noise to data in $T$ steps. The **noise schedule** $\beta_1, \ldots, \beta_T$ controls how much noise is added at each step. From the schedule we derive:

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

$\bar{\alpha}_t$ is a "slider" between data and noise:
- $\bar{\alpha}_t \approx 1$: mostly signal (early timesteps)
- $\bar{\alpha}_t \approx 0$: mostly noise (late timesteps)

Open `diffusion.py` and implement `linear_noise_schedule`.

In [ ]:
from diffusion import linear_noise_schedule

# Hyperparameters
T = 300
BETA_START = 1e-4
BETA_END = 0.02

noise_schedule = linear_noise_schedule(BETA_START, BETA_END, T)
alpha_bars = noise_schedule["alpha_bars"]

print(f"betas shape:      {noise_schedule['betas'].shape}")
print(f"alphas shape:     {noise_schedule['alphas'].shape}")
print(f"alpha_bars shape: {alpha_bars.shape}")
print(f"alpha_bars[0]:    {float(alpha_bars[0]):.4f}  (should be ~1)")
print(f"alpha_bars[-1]:   {float(alpha_bars[-1]):.4f}  (should be ~0)")

# Check monotonicity
assert np.all(np.diff(np.array(alpha_bars)) < 0), "alpha_bars should decrease"
print("\nNoise schedule check passed!")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ts = np.arange(T)
axes[0].plot(ts, noise_schedule["betas"], "b-")
axes[0].set(xlabel="Timestep $t$", ylabel=r"$\beta_t$", title="Noise Schedule")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts, alpha_bars, "r-")
axes[1].set(xlabel="Timestep $t$", ylabel=r"$\bar{\alpha}_t$", title="Cumulative Signal Fraction")
axes[1].axhline(0, color="gray", ls="--", alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part C: Diffusion Kernel (TODO #2)

The key insight from lecture: because each forward step is Gaussian, the composition is also Gaussian. This gives us the **diffusion kernel** — a closed-form expression for $q(z_t \mid x_0)$ that lets us jump directly to any timestep without running the chain:

$$q(z_t \mid x_0) = \mathcal{N}\!\left(z_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) I\right)$$

or equivalently, by the reparameterization trick:

$$z_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

Open `diffusion.py` and implement `q_sample`.

In [ ]:
from diffusion import q_sample

key = jr.PRNGKey(0)
x_batch = jnp.array(X_train[:8])
noise = jr.normal(key, x_batch.shape)

# At t=0: z_t should be very close to x_0
t_early = jnp.zeros(8, dtype=jnp.int32)
z_early = q_sample(x_batch, t_early, noise, alpha_bars)
print(f"t=0:   MSE to clean = {float(jnp.mean((z_early - x_batch)**2)):.6f} (should be ~0)")

# At t=T-1: z_t should be close to pure noise
t_late = jnp.full(8, T - 1, dtype=jnp.int32)
z_late = q_sample(x_batch, t_late, noise, alpha_bars)
print(f"t={T-1}: MSE to clean = {float(jnp.mean((z_late - x_batch)**2)):.4f} (should be large)")
print(f"t={T-1}: MSE to noise = {float(jnp.mean((z_late - noise)**2)):.4f} (should be small)")

assert z_early.shape == x_batch.shape
print("\nForward process check passed!")

In [ ]:
# Visualize the forward process: watch a galaxy dissolve into noise
timesteps_to_show = [0, 25, 50, 100, 150, 200, 250, T - 1]
x_single = jnp.array(X_train[:1])  # one galaxy
noise_single = jr.normal(jr.PRNGKey(42), x_single.shape)

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(16, 2.5))
for i, t_val in enumerate(timesteps_to_show):
    t_arr = jnp.array([t_val])
    z = q_sample(x_single, t_arr, noise_single, alpha_bars)
    axes[i].imshow(np.clip(np.array(z[0, :, :, 0]), 0, 1), cmap="gray")
    axes[i].set_title(f"t={t_val}\n$\\bar{{\\alpha}}$={float(alpha_bars[t_val]):.2f}", fontsize=10)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
plt.suptitle("Forward Process: Galaxy → Noise", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part D: Noise Predictor Network — `EpsNet` (TODO #3)

The diffusion model learns a single network $\epsilon_\theta(z_t, t)$ that takes a noisy image and its timestep, and predicts the noise that was added. The same network is shared across all $T$ timesteps — it uses $t$ as an additional input to know the noise level.

**Time conditioning:** embed $t$ using `sinusoidal_embedding` (given), then inject the resulting vector into intermediate conv features. Flax `Conv` uses `padding='SAME'` by default, so spatial dimensions are preserved throughout.

**Spatial structure:** look at the galaxy images above. Galaxy images are **rotation-symmetric but not translation-invariant** — structure concentrates at the center, not uniformly across the image. A standard CNN applies the same filter everywhere and has no sense of where it is in the image. Think carefully about how to encode this prior into your network. Your architecture choice will affect the last grading criterion.

Open `diffusion.py` and implement `EpsNet`.

In [ ]:
from diffusion import EpsNet, sinusoidal_embedding

model = EpsNet()
key = jr.PRNGKey(0)

dummy_z = jnp.ones((4, 32, 32, 1))
dummy_t = jnp.array([0, 50, 150, 299])

params = model.init(key, dummy_z, dummy_t)
pred = model.apply(params, dummy_z, dummy_t)

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"Input shape:      {dummy_z.shape}")
print(f"Output shape:     {pred.shape}")
print(f"Total parameters: {n_params:,}")

assert pred.shape == dummy_z.shape, f"Expected {dummy_z.shape}, got {pred.shape}"
print("\nNetwork shape check passed!")

---
## Part E: Training Loss (TODO #4)

The training loss is the denoising objective from lecture:

$$\mathcal{L} = \mathbb{E}_{t,\, x,\, \epsilon}\left[\|\epsilon - \epsilon_\theta(z_t, t)\|^2\right]$$

For each training step: sample a random timestep $t$, sample noise $\epsilon$, compute $z_t$ via the forward process, predict the noise, and take the MSE.

Open `diffusion.py` and implement `compute_loss`.

In [ ]:
from diffusion import compute_loss, train_step

key = jr.PRNGKey(0)
key, init_key, loss_key = jr.split(key, 3)

X_batch = jnp.array(X_train[:64])
dummy_t = jnp.zeros(64, dtype=jnp.int32)
params = model.init(init_key, X_batch, dummy_t)

loss = compute_loss(params, model, X_batch, loss_key, alpha_bars, T)
print(f"Loss: {float(loss):.4f}")

assert np.isfinite(float(loss)), "Loss is not finite"
assert float(loss) > 0, "Loss should be positive"
print("Loss function check passed!")

---
## Part F: Training Step (Given)

The `train_step` function is provided — it computes `compute_loss`, differentiates with `jax.value_and_grad`, and applies an optimizer update. Verify it works:

In [ ]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

losses = []
for i in range(20):
    key, step_key = jr.split(key)
    params, opt_state, loss = train_step(
        params, opt_state, X_batch, model, optimizer, step_key, alpha_bars, T
    )
    losses.append(float(loss))

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
assert losses[-1] < losses[0], f"Loss did not decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"
print("train_step check passed!")

---
## Part G: Train the Model

Train the diffusion model on the galaxy images. The first call triggers JIT compilation.

In [ ]:
def train_diffusion(X_train, noise_schedule, T, n_epochs=150, batch_size=256, lr=1e-3, seed=42):
    """Train a diffusion model and return params + loss history."""
    alpha_bars = noise_schedule["alpha_bars"]
    model = EpsNet()
    key = jr.PRNGKey(seed)
    key, init_key = jr.split(key)

    X_train_jax = jnp.array(X_train)
    dummy_t = jnp.zeros(1, dtype=jnp.int32)
    params = model.init(init_key, X_train_jax[:1], dummy_t)

    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)

    jit_train_step = jax.jit(
        lambda p, o, x, k: train_step(p, o, x, model, optimizer, k, alpha_bars, T)
    )

    n_train = len(X_train_jax)
    rng = np.random.default_rng(seed)
    history = []

    for epoch in range(n_epochs):
        idx = rng.permutation(n_train)
        epoch_losses = []

        for start in range(0, n_train, batch_size):
            batch_idx = idx[start : start + batch_size]
            key, step_key = jr.split(key)
            params, opt_state, loss = jit_train_step(
                params, opt_state, X_train_jax[batch_idx], step_key
            )
            epoch_losses.append(float(loss))

        mean_loss = np.mean(epoch_losses)
        history.append(mean_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:3d}/{n_epochs} — loss: {mean_loss:.4f}")

    return params, model, history

In [ ]:
params, model, history = train_diffusion(X_train, noise_schedule, T)

from diffusion import save_model
save_model(params, "model_params.pkl")
print("\nSaved model_params.pkl")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history) + 1), history, "b-o", markersize=3)
ax.set(xlabel="Epoch", ylabel="MSE Loss", title="Diffusion Training Loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Visualize: Forward Process

Watch several galaxies dissolve into noise as $t$ increases.

In [ ]:
timesteps_to_show = [0, 30, 75, 150, 225, T - 1]
n_show = 4
x_show = jnp.array(X_val[:n_show])
key, noise_key = jr.split(jr.PRNGKey(0))
noise_show = jr.normal(noise_key, x_show.shape)

fig, axes = plt.subplots(n_show, len(timesteps_to_show), figsize=(14, 6))
for col, t_val in enumerate(timesteps_to_show):
    t_arr = jnp.full(n_show, t_val)
    z = q_sample(x_show, t_arr, noise_show, alpha_bars)
    for row in range(n_show):
        axes[row, col].imshow(np.clip(np.array(z[row, :, :, 0]), 0, 1), cmap="gray")
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
    axes[0, col].set_title(f"t={t_val}", fontsize=11)
axes[0, 0].set_ylabel("Galaxies", fontsize=11)
plt.suptitle("Forward Diffusion: Signal → Noise", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part H: Generate New Galaxies

Now use the trained model to generate new galaxies by running the reverse process. The `ddpm_sample_step` function is provided — it implements one reverse step:

1. Predict noise: $\hat{\epsilon} = \epsilon_\theta(z_t, t)$
2. Compute mean: $\mu = \frac{1}{\sqrt{\alpha_t}}\left(z_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\hat{\epsilon}\right)$
3. Sample: $z_{t-1} = \mu + \sqrt{\beta_t}\, \eta$, where $\eta \sim \mathcal{N}(0, I)$ (skip at $t = 0$)

Starting from pure noise $z_T \sim \mathcal{N}(0, I)$, loop from $t = T-1$ down to $t = 0$.

In [ ]:
from diffusion import ddpm_sample_step

def generate_samples(model, params, noise_schedule, n_samples=16, seed=0):
    """Generate samples by running the full reverse process."""
    T = len(noise_schedule["betas"])
    key = jax.random.PRNGKey(seed)
    z = jax.random.normal(key, (n_samples, 32, 32, 1))

    for t in range(T - 1, -1, -1):
        key, step_key = jax.random.split(key)
        z = ddpm_sample_step(model, params, z, t, noise_schedule, step_key)

    return z

print("Generating samples (this runs the reverse process for all T steps)...")
samples = generate_samples(model, params, noise_schedule, n_samples=16)
print(f"Sample shape: {samples.shape}")
print(f"Sample range: [{float(samples.min()):.3f}, {float(samples.max()):.3f}]")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(np.array(samples[i, :, :, 0]), 0, 1), cmap="gray")
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Generated Galaxies (from noise)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Watching the Denoising Process

One of the beautiful aspects of diffusion: we can watch the generation unfold step by step. Structure emerges gradually from noise — coarse features first, then fine details.

In [ ]:
# Run reverse process and save snapshots
snapshot_steps = [T - 1, int(0.75 * T), int(0.5 * T), int(0.25 * T), int(0.1 * T), 0]
n_show = 4
key = jr.PRNGKey(123)
z = jr.normal(key, (n_show, 32, 32, 1))
snapshots = {}

for t in range(T - 1, -1, -1):
    if t in snapshot_steps:
        snapshots[t] = np.array(z)
    key, step_key = jr.split(key)
    z = ddpm_sample_step(model, params, z, t, noise_schedule, step_key)
snapshots[-1] = np.array(z)  # final output (after t=0 step)

# Plot
cols = snapshot_steps + [-1]
fig, axes = plt.subplots(n_show, len(cols), figsize=(14, 6))
for col_idx, t_val in enumerate(cols):
    for row in range(n_show):
        axes[row, col_idx].imshow(
            np.clip(snapshots[t_val][row, :, :, 0], 0, 1), cmap="gray"
        )
        axes[row, col_idx].set_xticks([])
        axes[row, col_idx].set_yticks([])
    label = f"t={t_val}" if t_val >= 0 else "Final"
    axes[0, col_idx].set_title(label, fontsize=11)

plt.suptitle("Reverse Process: Noise → Galaxy", fontsize=14)
plt.tight_layout()
plt.show()

---
## Real vs Generated

Compare real galaxies (top) with generated ones (bottom).

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))

rng = np.random.default_rng(0)
real_idx = rng.choice(len(X_val), 8, replace=False)
for i in range(8):
    axes[0, i].imshow(X_val[real_idx[i], :, :, 0], cmap="gray")
    axes[1, i].imshow(np.clip(np.array(samples[i, :, :, 0]), 0, 1), cmap="gray")
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel("Real", fontsize=12, rotation=90, labelpad=10)
axes[1, 0].set_ylabel("Generated", fontsize=12, rotation=90, labelpad=10)
plt.suptitle("Real vs Generated Galaxies", fontsize=14)
plt.tight_layout()
plt.show()

---
## Self-Check

This replicates the autograder. Run before submitting.

In [ ]:
from diffusion import load_model as _load_model

print("Running autograder checks...\n")

# Check 1: Noise schedule + forward process
try:
    _ns = linear_noise_schedule(1e-4, 0.02, 300)
    assert _ns["betas"].shape == (300,)
    assert _ns["alphas"].shape == (300,)
    assert _ns["alpha_bars"].shape == (300,)
    assert np.all(np.diff(np.array(_ns["alpha_bars"])) < 0)
    assert float(_ns["alpha_bars"][0]) > 0.99
    assert float(_ns["alpha_bars"][-1]) < 0.5
    _x = jr.uniform(jr.PRNGKey(0), (8, 32, 32, 1))
    _noise = jr.normal(jr.PRNGKey(1), _x.shape)
    _z_early = q_sample(_x, jnp.zeros(8, dtype=jnp.int32), _noise, _ns["alpha_bars"])
    _z_late = q_sample(_x, jnp.full(8, 299, dtype=jnp.int32), _noise, _ns["alpha_bars"])
    assert _z_early.shape == _x.shape
    assert float(jnp.mean((_z_early - _x) ** 2)) < 0.01
    assert float(jnp.mean((_z_late - _noise) ** 2)) < float(jnp.mean((_z_late - _x) ** 2))
    print("[PASS] (1 pt) Noise schedule + forward process")
except Exception as e:
    print(f"[FAIL] (1 pt) Noise schedule + forward process: {e}")

# Check 2: Loss decreases
if losses[-1] < losses[0]:
    print(f"[PASS] (1 pt) Loss decreased: {losses[0]:.4f} -> {losses[-1]:.4f}")
else:
    print(f"[FAIL] (1 pt) Loss did not decrease: {losses[0]:.4f} -> {losses[-1]:.4f}")

# Check 3 & 4: Sample quality + radial structure
try:
    _params = _load_model("model_params.pkl")
    _ns = linear_noise_schedule(1e-4, 0.02, 300)
    _key = jr.PRNGKey(42)
    _z = jr.normal(_key, (16, 32, 32, 1))
    for _t in range(299, -1, -1):
        _key, _sk = jr.split(_key)
        _z = ddpm_sample_step(model, _params, _z, _t, _ns, _sk)
    _s = np.array(_z)

    # Quality
    assert np.all(np.isfinite(_s)), "Samples contain NaN/Inf"
    _clip_diff = np.mean(np.abs(_s - np.clip(_s, 0, 1)))
    assert _clip_diff < 0.1, f"Samples far from [0,1] (clip_diff={_clip_diff:.3f})"
    assert float(np.std(_s)) > 0.01, "Samples have near-zero variance"
    print("[PASS] (1 pt) Sample quality: finite, near [0,1], varied")

    # Radial structure
    _s_clip = np.clip(_s, 0, 1)
    _center = _s_clip[:, 10:22, 10:22, 0]
    _edge_mask = np.ones((32, 32), dtype=bool)
    _edge_mask[4:28, 4:28] = False
    _edge = _s_clip[:, :, :, 0][:, _edge_mask]
    _ratio = float(np.mean(_center)) / (float(np.mean(_edge)) + 1e-8)
    if _ratio > 2.5:
        print(f"[PASS] (1 pt) Radial structure: center/edge ratio = {_ratio:.2f} > 2.5")
    else:
        print(
            f"[FAIL] (1 pt) Radial structure: center/edge ratio = {_ratio:.2f} < 2.5. "
            f"Consider adding spatial information (e.g. radial distance) to your network."
        )
except Exception as e:
    print(f"[FAIL] (1-2 pts) Sample generation failed: {e}")